# RAG Agent — TensorTonic Project

A complete RAG pipeline using:

- LangChain
- Chroma
- OpenAI-compatible LLM endpoint
- OpenAI-compatible embeddings
- Recursive character chunking
- Similarity retrieval
- Context-grounded generation


## 1. Install dependencies

In [ ]:
%pip install -q langchain langchain-core langchain-openai langchain-text-splitters langchain-chroma chromadb

## 2. Configuration

In [ ]:
import os

# Set this to the OpenAI-compatible endpoint used by the project.
# Example:
# os.environ["OPENAI_BASE_URL"] = "https://..."

if "OPENAI_BASE_URL" not in os.environ:
    raise RuntimeError(
        "Please set the OPENAI_BASE_URL environment variable first."
    )

SOURCE_PATH = "document.txt"

QUESTION = (
    "How long is customer data retained after workspace cancellation?"
)

print("Configuration loaded.")

## 3. Load the source document

In [ ]:
from langchain_core.documents import Document

with open(SOURCE_PATH, "r", encoding="utf-8") as f:
    text = f.read()

documents = [
    Document(
        page_content=text,
        metadata={"source": SOURCE_PATH},
    )
]

print(f"Loaded {len(text):,} characters from {SOURCE_PATH}")

## 4. Create the LLM

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="google.gemma-3-4b-it",
    base_url=os.environ["OPENAI_BASE_URL"],
)

print("LLM initialized.")

## 5. Create the embedding model

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="amazon.titan-embed-text-v2:0",
    base_url=os.environ["OPENAI_BASE_URL"],
    check_embedding_ctx_length=False,
)

print("Embedding model initialized.")

## 6. Split the document into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=60,
    add_start_index=True,
)

chunks = splitter.split_documents(documents)

for index, chunk in enumerate(chunks, start=1):
    chunk.metadata["chunk_id"] = f"chunk-{index}"

print(f"Created {len(chunks)} chunks.")

for chunk in chunks:
    preview = chunk.page_content[:120].replace("\n", " ").strip()
    print(f"{chunk.metadata['chunk_id']}: {preview}...")

## 7. Create the Chroma vector store

In [ ]:
from uuid import uuid4
from langchain_chroma import Chroma

collection_name = f"northstar-support-{uuid4().hex[:8]}"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=collection_name,
)

print(f"Indexed {len(chunks)} chunks in {collection_name}.")

## 8. Retrieval

In [ ]:
def retrieve(query: str, k: int = 3):
    """Return the k most similar chunks and their scores."""
    return vector_store.similarity_search_with_score(
        query,
        k=k,
    )


hits = retrieve(QUESTION, k=3)

print("Retrieved chunks:\n")

for rank, (document, distance) in enumerate(hits, start=1):
    chunk_id = document.metadata["chunk_id"]
    preview = document.page_content[:160].replace("\n", " ").strip()

    print(f"{rank}. {chunk_id} | distance={distance:.4f}")
    print(f"   {preview}...\n")

## 9. Build the retrieved context

In [ ]:
def build_context(retrieved_hits) -> str:
    blocks = []

    for document, _distance in retrieved_hits:
        chunk_id = document.metadata["chunk_id"]

        blocks.append(
            f"[{chunk_id}]\n"
            f"{document.page_content}"
        )

    return "\n\n".join(blocks)


context = build_context(hits)

print("=" * 70)
print(context)
print("=" * 70)

## 10. RAG system prompt

In [ ]:
SYSTEM_PROMPT = """
You answer support questions using only the provided context.

Rules:

1. Use only information present in the context.
2. Do not invent facts.
3. Cite every factual claim using the chunk ID.
4. Use citations such as [chunk-2].
5. If the context does not contain the answer, say:

   I do not know based on the provided context.

Keep the answer concise and directly answer the question.
"""

## 11. Complete RAG function

In [ ]:
def answer_with_rag(query: str, k: int = 3) -> dict:
    # Retrieve
    retrieved_hits = retrieve(query, k=k)

    # Build context
    context = build_context(retrieved_hits)

    # Construct prompt
    messages = [
        ("system", SYSTEM_PROMPT),
        (
            "human",
            f"""
Context:

{context}

Question:

{query}
""",
        ),
    ]

    # Generate answer
    response = llm.invoke(messages)

    # Collect source IDs
    source_ids = [
        document.metadata["chunk_id"]
        for document, _distance in retrieved_hits
    ]

    return {
        "answer": response.content,
        "source_ids": source_ids,
        "retrieved_documents": [
            document for document, _distance in retrieved_hits
        ],
    }

## 12. Ask a question

In [ ]:
result = answer_with_rag(QUESTION, k=3)

print("=" * 70)
print("FINAL ANSWER")
print("=" * 70)
print(result["answer"])

print("\nRetrieved sources:")
for source_id in result["source_ids"]:
    print(f"- {source_id}")

## 13. Ask your own questions

In [ ]:
query = input("Question: ")

result = answer_with_rag(query, k=3)

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
for source_id in result["source_ids"]:
    print(f"- {source_id}")